In [ ]:


#RAG: Allowing agent to give solutions using external resources or knowledge given
#Abig document you can divide the document in small chunks 
#Concept is found by embeddings: Converting text into vector and store them in a vector database: Using indexing
#Retrieval:You will generate text using the embeddings

#Benefits of RAG - Accurate answers and cost effective 

#All RAG applications require vector database like "Chroma"

import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "Telecom_RAG_Learning_Document.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded{len(pages)} from the PDF)")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

In [ ]:
#Creating chunks: we use overlap by taking a little information from one page to the next page and like that
#This is used when we have a very big document oftenly


from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "],
)

chunks = splitter.split_documents(pages)

len(chunks)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="telecom_rag_clean"
)

print("Chunks:", len(chunks))
print("Vectors in Chroma:", vector_store._collection.count())

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

test_query = "Tell me about RAG and what it does"

retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content[:300])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# Format retrieved documents into plain text
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using only the context provided below.
If the context does not contain enough information, say so clearly.

Context
{context}
"""
prompt = ChatPromptTemplate.from_messages([
("system", SYSTEM_PROMPT),
("human", "{question}"),
])

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0,
    reasoning_format = "parsed",
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    |StrOutputParser()
)
print("RAG chain assembled.")

In [ ]:
question = "How does mobile payment work and issues related to it"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))